In [1]:
import polars as pl

df = pl.scan_parquet("../data/raw/wikipedia_indonesia.parquet")

# Hapus duplikasi berdasarkan kolom id/title jika ada
df_clean = df.unique(subset=["docid"])

# Cek null values pada judul/teks
null_count = df_clean.select([
    pl.col("title").null_count().alias("null_title"),
    pl.col("text").null_count().alias("null_text")
]).collect()
print(null_count)

shape: (1, 2)
┌────────────┬───────────┐
│ null_title ┆ null_text │
│ ---        ┆ ---       │
│ u32        ┆ u32       │
╞════════════╪═══════════╡
│ 0          ┆ 0         │
└────────────┴───────────┘


In [2]:
import duckdb

# Jalankan query SUMMARIZE langsung dari file parquet
duckdb.sql("""
    SUMMARIZE SELECT docid, title, length(text) as text_length 
    FROM read_parquet('../data/raw/wikipedia_indonesia.parquet')
""").show()

┌─────────────┬─────────────┬─────────────┬──────────┬───────────────┬────────────────────┬───────────────────┬─────────┬─────────┬─────────┬─────────┬─────────────────┐
│ column_name │ column_type │     min     │   max    │ approx_unique │        avg         │        std        │   q25   │   q50   │   q75   │  count  │ null_percentage │
│   varchar   │   varchar   │   varchar   │ varchar  │     int64     │      varchar       │      varchar      │ varchar │ varchar │ varchar │  int64  │  decimal(9,2)   │
├─────────────┼─────────────┼─────────────┼──────────┼───────────────┼────────────────────┼───────────────────┼─────────┼─────────┼─────────┼─────────┼─────────────────┤
│ docid       │ VARCHAR     │ 1#1         │ 999993#1 │       1361830 │ NULL               │ NULL              │ NULL    │ NULL    │ NULL    │ 1301683 │            0.00 │
│ title       │ VARCHAR     │ !!! (album) │ ㅑ       │        396286 │ NULL               │ NULL              │ NULL    │ NULL    │ NULL    │ 1301683 │

In [3]:
# Simpan data bersih ke format parquet di data/processed/
df_clean.sink_parquet("../data/processed/wikipedia_clean.parquet")